# Cluster & Culture — Plotly Visualizations

Ports the three main views from `main.py` into interactive Plotly graphs:
1. **PCA Scatter** (2D & 3D)
2. **Custom-Axis Scatter**
3. **Confusion Matrix** (classifier evaluation)

In [185]:
# %pip install plotly  # uncomment if needed

import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"

df_raw = pd.read_csv(DATA_DIR / "llm_benchmarks.csv")
df_raw["Company"]      = df_raw["Model"].str.split("/").str[0]
df_raw["Release Date"] = pd.to_datetime(df_raw["Release Date"], errors="coerce")
df_raw["Release Year"] = df_raw["Release Date"].dt.year.astype("Int64")

with open(DATA_DIR / "column_metadata.json") as f:
    col_meta = json.load(f)

BENCHMARK_COLS = [
    col for col, meta in col_meta["columns"].items()
    if meta.get("is_benchmark") and col in df_raw.columns
]

PRIMARY_BENCHMARKS = [
    "oll_BBH", "oll_GPQA", "oll_IFEval", "oll_MATH_lvl5", "oll_MMLU_PRO", "oll_MUSR",
]

print(f"Loaded {len(df_raw)} models, {len(BENCHMARK_COLS)} benchmark columns")
df_raw.head(3)

Loaded 4595 models, 20 benchmark columns


,Model,Release Date,Origin,oe_aggregate,oe_aime2026,oe_coverage_count,oe_coverage_percent,oe_evasionBench,oe_gpqa,oe_gsm8k,...,oll_CO2_kg,oll_GPQA,oll_IFEval,oll_MATH_lvl5,oll_MMLU_PRO,oll_MUSR,oll_average,arena_text_elo,Company,Release Year
0,0-hero/Matter-0.2-7B-DPO,2024-04-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.219174,1.230425,33.027921,1.435045,1.817376,5.871875,8.906361,NaN,0-hero,2024
1,01-ai/Yi-1.5-34B,2024-05-11,China,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,22.703398,15.436242,28.411725,15.332326,40.732122,11.217188,25.646494,NaN,01-ai,2024
2,01-ai/Yi-1.5-34B-32K,2024-05-15,China,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,23.154629,15.100671,31.186917,15.407855,41.212323,14.077865,26.727913,NaN,01-ai,2024


## 1. PCA Scatter

In [186]:
def run_pca(dims=2, benchmarks=None, min_coverage=0.5):
    selected = benchmarks or PRIMARY_BENCHMARKS[:]
    selected = [b for b in selected if b in df_raw.columns]

    df = df_raw.copy()
    coverage = df[selected].notna().mean(axis=1)
    df = df[coverage >= min_coverage].copy()

    X = df[selected].values.astype(float)
    X = SimpleImputer(strategy="mean").fit_transform(X)
    X = StandardScaler().fit_transform(X)

    n_comp = min(dims, X.shape[1], X.shape[0])
    pca = PCA(n_components=n_comp, random_state=42)
    coords = pca.fit_transform(X)

    df["PC1"] = coords[:, 0]
    df["PC2"] = coords[:, 1]
    if dims == 3 and n_comp >= 3:
        df["PC3"] = coords[:, 2]

    var = pca.explained_variance_ratio_
    return df, var

# ── 2D PCA ────────────────────────────────────────────────────────────────────
df_pca, var = run_pca(dims=2)

fig_pca2d = px.scatter(
    df_pca,
    x="PC1", y="PC2",
    color="Origin",
    hover_name="Model",
    hover_data={"Company": True, "Release Year": True, "PC1": ":.3f", "PC2": ":.3f"},
    title=(
        f"PCA 2D — LLM Benchmarks<br>"
        f"<sup>PC1 {var[0]:.1%} var | PC2 {var[1]:.1%} var</sup>"
    ),
    labels={"PC1": f"PC1 ({var[0]:.1%})", "PC2": f"PC2 ({var[1]:.1%})"},
    template="plotly_white",
    width=1280, height=720,
)
fig_pca2d.update_traces(marker=dict(size=8, opacity=0.85))
fig_pca2d.show()

In [187]:
# ── 3D PCA ────────────────────────────────────────────────────────────────────
df_pca3, var3 = run_pca(dims=3)

fig_pca3d = px.scatter_3d(
    df_pca3,
    x="PC1", y="PC2", z="PC3",
    color="Origin",
    hover_name="Model",
    hover_data={"Company": True, "Release Year": True},
    title=(
        f"PCA 3D — LLM Benchmarks<br>"
        f"<sup>PC1 {var3[0]:.1%} | PC2 {var3[1]:.1%} | PC3 {var3[2]:.1%}</sup>"
    ),
    labels={
        "PC1": f"PC1 ({var3[0]:.1%})",
        "PC2": f"PC2 ({var3[1]:.1%})",
        "PC3": f"PC3 ({var3[2]:.1%})",
    },
    template="plotly_white",
    width=1280, height=720,
)
fig_pca3d.update_traces(marker=dict(size=5, opacity=0.85))
fig_pca3d.show()

## 1b. PCA — Principal Component Makeup (Loadings)

In [188]:
# Re-run PCA and extract loadings for all primary benchmarks
selected = PRIMARY_BENCHMARKS[:]
df_load = df_raw.copy()
coverage = df_load[selected].notna().mean(axis=1)
df_load = df_load[coverage >= 0.5]

X_load = df_load[selected].values.astype(float)
X_load = SimpleImputer(strategy="mean").fit_transform(X_load)
X_load = StandardScaler().fit_transform(X_load)

pca_full = PCA(n_components=min(3, len(selected)), random_state=42)
pca_full.fit(X_load)
var_full = pca_full.explained_variance_ratio_

# ── Grouped bar: each benchmark's loading across PC1/PC2/PC3 ─────────────────
n_components = pca_full.n_components_
df_loadings = pd.DataFrame(
    pca_full.components_[:n_components].T,
    index=selected,
    columns=[f"PC{i+1} ({var_full[i]:.1%})" for i in range(n_components)],
).reset_index().rename(columns={"index": "Benchmark"})

df_melt = df_loadings.melt(id_vars="Benchmark", var_name="Component", value_name="Loading")

fig_load = px.bar(
    df_melt,
    x="Benchmark", y="Loading", color="Component",
    barmode="group",
    title="PCA Loadings — Benchmark Contribution per Principal Component",
    template="plotly_white",
    width=1280, height=720,
)
fig_load.add_hline(y=0, line_color="black", line_width=0.5)
fig_load.update_layout(xaxis_tickangle=-30)
fig_load.show()

# ── Heatmap view (easier to read with many benchmarks) ───────────────────────
fig_load_heat = go.Figure(go.Heatmap(
    z=pca_full.components_[:n_components],
    x=selected,
    y=[f"PC{i+1} ({var_full[i]:.1%})" for i in range(n_components)],
    colorscale="RdBu",
    zmid=0,
    text=np.round(pca_full.components_[:n_components], 3),
    texttemplate="%{text}",
    hovertemplate="Benchmark: %{x}<br>Component: %{y}<br>Loading: %{z:.3f}<extra></extra>",
))
fig_load_heat.update_layout(
    title="PCA Loadings Heatmap",
    xaxis_title="Benchmark",
    yaxis_title="Principal Component",
    template="plotly_white",
    xaxis_tickangle=-30,
    width=1280, height=720,
)
fig_load_heat.show()

## 1c. UMAP

Note: UMAP is designed for high-dimensional data. With only 6 benchmark features the topology signal is limited — treat this as exploratory. Increase `N_NEIGHBORS` for more global structure, decrease for more local clusters.

In [189]:
import umap

N_NEIGHBORS  = 10    # higher = more global structure
MIN_DIST     = 0.1   # lower = tighter clusters
RANDOM_STATE = 42

selected_umap = PRIMARY_BENCHMARKS[:]
df_umap = df_raw.copy()
coverage = df_umap[selected_umap].notna().mean(axis=1)
df_umap = df_umap[coverage >= 0.5].copy()

X_umap = df_umap[selected_umap].values.astype(float)
X_umap = SimpleImputer(strategy="mean").fit_transform(X_umap)
X_umap = StandardScaler().fit_transform(X_umap)

reducer = umap.UMAP(n_components=2, n_neighbors=N_NEIGHBORS, min_dist=MIN_DIST,
                    random_state=RANDOM_STATE)
embedding = reducer.fit_transform(X_umap)

df_umap["UMAP1"] = embedding[:, 0]
df_umap["UMAP2"] = embedding[:, 1]

fig_umap = px.scatter(
    df_umap,
    x="UMAP1", y="UMAP2",
    color="Origin",
    hover_name="Model",
    hover_data={
        "Company": True,
        "Release Year": True,
        "UMAP1": False,
        "UMAP2": False,
        **{b: ":.2f" for b in PRIMARY_BENCHMARKS},
    },
    title=f"UMAP 2D — LLM Benchmarks<br><sup>n_neighbors={N_NEIGHBORS}, min_dist={MIN_DIST}</sup>",
    template="plotly_white",
    width=1280, height=720,
)
fig_umap.update_traces(marker=dict(size=8, opacity=0.85))
fig_umap.show()

C:\Users\aidan\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



## 1d. UMAP — Benchmark Correlation with Dimensions

UMAP is nonlinear so there are no true loadings. Instead: Spearman correlation of each benchmark with UMAP1/UMAP2. Analogous read to 1b — shows which benchmarks align with each axis, but correlation not causation.

In [190]:
from scipy.stats import spearmanr

# df_umap and embedding must exist — run 1c first
corr_rows = []
for bench in PRIMARY_BENCHMARKS:
    vals = df_umap[bench].values
    mask = ~np.isnan(vals)
    r1, _ = spearmanr(vals[mask], df_umap["UMAP1"].values[mask])
    r2, _ = spearmanr(vals[mask], df_umap["UMAP2"].values[mask])
    corr_rows.append({"Benchmark": bench, "UMAP1": round(r1, 3), "UMAP2": round(r2, 3)})

df_corr = pd.DataFrame(corr_rows)
df_corr_melt = df_corr.melt(id_vars="Benchmark", var_name="Dimension", value_name="Spearman r")

# ── Grouped bar ───────────────────────────────────────────────────────────────
fig_ucorr = px.bar(
    df_corr_melt,
    x="Benchmark", y="Spearman r", color="Dimension",
    barmode="group",
    title="UMAP Dimension Correlation — Spearman r per Benchmark",
    template="plotly_white",
    width=1280, height=720,
)
fig_ucorr.add_hline(y=0, line_color="black", line_width=0.5)
fig_ucorr.update_layout(xaxis_tickangle=-30, yaxis_range=[-1, 1])
fig_ucorr.show()

# ── Heatmap ───────────────────────────────────────────────────────────────────
fig_ucorr_heat = go.Figure(go.Heatmap(
    z=df_corr[["UMAP1", "UMAP2"]].values.T,
    x=df_corr["Benchmark"].tolist(),
    y=["UMAP1", "UMAP2"],
    colorscale="RdBu",
    zmid=0, zmin=-1, zmax=1,
    text=df_corr[["UMAP1", "UMAP2"]].values.T,
    texttemplate="%{text}",
    hovertemplate="Benchmark: %{x}<br>Dimension: %{y}<br>Spearman r: %{z:.3f}<extra></extra>",
))
fig_ucorr_heat.update_layout(
    title="UMAP Dimension Correlation Heatmap",
    xaxis_title="Benchmark",
    yaxis_title="UMAP Dimension",
    template="plotly_white",
    xaxis_tickangle=-30,
    width=1280, height=720,
)
fig_ucorr_heat.show()

## 2. Custom-Axis Scatter

Change `X_AXIS` / `Y_AXIS` to any benchmark column name.

In [191]:
X_AXIS = "oll_IFEval"
Y_AXIS = "oll_MATH_lvl5"
COLOR_BY = "Origin"   # "Origin" | "Company" | "Release Year"

df_scatter = df_raw[[X_AXIS, Y_AXIS, "Model", "Company", "Origin", "Release Year"]].dropna(
    subset=[X_AXIS, Y_AXIS]
)

fig_scatter = px.scatter(
    df_scatter,
    x=X_AXIS, y=Y_AXIS,
    color=COLOR_BY,
    hover_name="Model",
    hover_data={"Company": True, "Origin": True, "Release Year": True,
                X_AXIS: ":.2f", Y_AXIS: ":.2f"},
    title=f"{Y_AXIS} vs {X_AXIS}",
    # trendline="ols",
    template="plotly_white",
    width=1280, height=720,
)
fig_scatter.update_traces(marker=dict(size=9, opacity=0.8))
fig_scatter.show()

## 3. Classifier + Confusion Matrix

Change `CLASSIFIER` (`"svm"` / `"dt"` / `"lr"`) and `TARGET` (`"origin"` / `"company"` / `"release_year"`).

In [192]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

CLASSIFIER   = "svm"      # "svm" | "dt" | "lr"
TARGET       = "origin"   # "origin" | "company" | "release_year"
TOP_N        = 8
MIN_COVERAGE = 0.5

# ── Country filter ────────────────────────────────────────────────────────────
# Set to None to include all countries, or list any subset of Origin values.
# Examples:
#   ORIGINS = None                                          # all countries
#   ORIGINS = ["United States", "China"]                    # just these two
#   ORIGINS = ["United States", "China", "France", "United Kingdom"]
ORIGINS = ["United States", "China"]
# ─────────────────────────────────────────────────────────────────────────────

TARGET_MAP = {
    "origin":       ("Origin",       "Origin / Country"),
    "company":      ("Company",      "Company"),
    "release_year": ("Release Year", "Release Year"),
}
col_name, target_label = TARGET_MAP[TARGET]

df_clf = df_raw.copy()
coverage = df_clf[PRIMARY_BENCHMARKS].notna().mean(axis=1)
df_clf = df_clf[coverage >= MIN_COVERAGE]

if ORIGINS is not None:
    df_clf = df_clf[df_clf["Origin"].isin(ORIGINS)]

y_raw = df_clf[col_name].fillna("Unknown").astype(str)
top_classes = set(y_raw.value_counts().head(TOP_N).index)
mask = y_raw.isin(top_classes)
y_raw, df_clf = y_raw[mask], df_clf[mask]
class_counts = y_raw.value_counts()
valid = set(class_counts[class_counts >= 3].index)
mask2 = y_raw.isin(valid)
y_raw, df_clf = y_raw[mask2], df_clf[mask2]

X = df_clf[PRIMARY_BENCHMARKS].values.astype(float)
X = SimpleImputer(strategy="mean").fit_transform(X)
X = StandardScaler().fit_transform(X)
y = y_raw.values
labels = sorted(y_raw.unique().tolist())

try:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
except ValueError:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

# Use probability-capable classifiers for ROC
clf_map = {
    "svm": (SVC(kernel="rbf", C=1.0, random_state=42, probability=True), "SVM (RBF)"),
    "dt":  (DecisionTreeClassifier(max_depth=5, random_state=42),          "Decision Tree"),
    "lr":  (LogisticRegression(max_iter=2000, random_state=42),            "Logistic Regression"),
}
clf, clf_name = clf_map[CLASSIFIER]
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
y_prob = clf.predict_proba(X_te)
acc = accuracy_score(y_te, y_pred)
cm = confusion_matrix(y_te, y_pred, labels=labels)

origin_label = f"{', '.join(ORIGINS)}" if ORIGINS else "All"
print(f"{clf_name} | target: {target_label} | origins: {origin_label} | accuracy: {acc:.1%} ({len(X_te)} test samples)")

SVM (RBF) | target: Origin / Country | origins: United States, China | accuracy: 87.1% (497 test samples)


In [193]:
# ── Confusion matrix heatmap ──────────────────────────────────────────────────
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig_cm = go.Figure(go.Heatmap(
    z=cm_norm,
    x=labels,
    y=labels,
    colorscale="Blues",
    zmin=0, zmax=1,
    text=cm,
    texttemplate="%{text}",
    hovertemplate="True: %{y}<br>Pred: %{x}<br>Count: %{text}<br>Rate: %{z:.1%}<extra></extra>",
))
fig_cm.update_layout(
    title=f"Confusion Matrix — {clf_name} predicting {target_label}<br><sup>Accuracy: {acc:.1%}</sup>",
    xaxis_title="Predicted",
    yaxis_title="True",
    template="plotly_white",
    yaxis_autorange="reversed",
    width=900, height=720,
)
fig_cm.show()

In [194]:
# ── ROC AUC Curve ─────────────────────────────────────────────────────────────
y_te_bin = label_binarize(y_te, classes=labels)

fig_roc = go.Figure()

# Per-class ROC
aucs = {}
if len(labels) == 2:
    # Binary case: single curve
    fpr, tpr, _ = roc_curve(y_te_bin[:, 0], y_prob[:, 0] if y_prob.shape[1] == 1 else y_prob[:, 1])
    roc_auc = auc(fpr, tpr)
    aucs[labels[1]] = roc_auc
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines",
        name=f"{labels[1]} vs {labels[0]} (AUC={roc_auc:.3f})",
    ))
else:
    # Multiclass: one-vs-rest per class
    for i, lbl in enumerate(labels):
        fpr, tpr, _ = roc_curve(y_te_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        aucs[lbl] = roc_auc
        fig_roc.add_trace(go.Scatter(
            x=fpr, y=tpr, mode="lines",
            name=f"{lbl} (AUC={roc_auc:.3f})",
        ))

# Diagonal reference
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines",
    line=dict(dash="dash", color="gray"),
    name="Random (AUC=0.500)", showlegend=True,
))

macro_auc = np.mean(list(aucs.values()))
fig_roc.update_layout(
    title=f"ROC Curve — {clf_name} predicting {target_label}<br><sup>Macro AUC: {macro_auc:.3f}</sup>",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
    width=800, height=800,
    xaxis=dict(constrain="domain", scaleanchor="y"),
    yaxis=dict(constrain="domain"),
    legend=dict(x=0.6, y=0.05),
)
fig_roc.show()

In [195]:
# ── Feature importances (DT / LR only) ───────────────────────────────────────
if CLASSIFIER == "dt":
    importances = dict(zip(PRIMARY_BENCHMARKS, clf.feature_importances_))
elif CLASSIFIER == "lr" and hasattr(clf, "coef_"):
    importances = dict(zip(PRIMARY_BENCHMARKS, np.abs(clf.coef_).mean(axis=0)))
else:
    importances = None

if importances:
    df_imp = pd.DataFrame(list(importances.items()), columns=["Benchmark", "Importance"])
    df_imp = df_imp.sort_values("Importance", ascending=True)
    fig_imp = px.bar(
        df_imp, x="Importance", y="Benchmark", orientation="h",
        title=f"Feature Importances — {clf_name}",
        template="plotly_white",
        width=1280, height=720,
    )
    fig_imp.show()
else:
    print("SVM has no native feature importances — skipping.")

SVM has no native feature importances — skipping.


---

# ECI (Epoch Composite Index) Data

Same analysis pipeline but on the ECI benchmark dataset. This data is much sparser — 59 benchmarks across 270 models with uneven coverage. Use `TOP_K_BENCHMARKS` below to pick how many benchmarks to include (4–10); benchmarks are ranked by how many models have a score.

In [196]:
# ── ECI Data Load & Benchmark Selection ───────────────────────────────────────
from itertools import combinations

TOP_K_BENCHMARKS = 4  # choose 4–10

# Load and pivot to wide format
df_eci_scores = pd.read_csv(DATA_DIR / "eci_scores.csv")
df_eci_raw = pd.read_csv(DATA_DIR / "processed_data_for_eci.csv")

df_eci_wide = df_eci_raw.pivot_table(
    index="Model", columns="benchmark", values="performance", aggfunc="first"
)

# Candidate pool: top 15 benchmarks by individual coverage
benchmark_coverage = df_eci_wide.notna().sum().sort_values(ascending=False)
candidate_benchmarks = benchmark_coverage.head(15).index.tolist()

# Find the combination of TOP_K_BENCHMARKS that maximizes models with 100% coverage
best_n, best_combo = 0, None
for combo in combinations(candidate_benchmarks, TOP_K_BENCHMARKS):
    n = df_eci_wide[list(combo)].dropna().shape[0]
    if n > best_n:
        best_n = n
        best_combo = combo

ECI_BENCHMARKS = list(best_combo)

print(f"Selected {TOP_K_BENCHMARKS} benchmarks (optimized for max 100% coverage):")
for b in ECI_BENCHMARKS:
    print(f"  {b}: {benchmark_coverage[b]} models individually")
print(f"\n→ {best_n} models have scores for ALL {TOP_K_BENCHMARKS} benchmarks")

# Merge metadata from eci_scores
df_eci = df_eci_wide[ECI_BENCHMARKS].copy()
df_eci = df_eci.merge(
    df_eci_scores[["Model", "eci", "date", "Organization", "Country (of organization)", "Accessibility group"]],
    left_index=True, right_on="Model", how="left",
)
df_eci["date"] = pd.to_datetime(df_eci["date"], errors="coerce")
df_eci["Release Year"] = df_eci["date"].dt.year.astype("Int64")
df_eci = df_eci.rename(columns={"Country (of organization)": "Country", "Accessibility group": "Access"})

# Require 100% coverage — only models with scores for ALL selected benchmarks
df_eci = df_eci.dropna(subset=ECI_BENCHMARKS).copy()

print(f"{len(df_eci)} models retained")
df_eci.head(3)

Selected 4 benchmarks (optimized for max 100% coverage):
  GPQA diamond: 183 models individually
  OTIS Mock AIME 2024-2025: 174 models individually
  DTBench: 158 models individually
  LMCA: 129 models individually

→ 111 models have scores for ALL 4 benchmarks
111 models retained


,GPQA diamond,OTIS Mock AIME 2024-2025,DTBench,LMCA,Model,eci,date,Organization,Country,Access,Release Year
199,0.156951,0.017073,0.168833,0.104000,Claude 3 Haiku,118.85,2024-03-07,Anthropic,United States of America,Closed weights,2024
175,0.307765,0.046268,0.360000,0.200000,Claude 3 Opus,126.90,2024-02-29,Anthropic,United States of America,Closed weights,2024
8,0.845258,1.000000,0.973333,0.719412,Claude Fable 5,163.42,2026-06-09,Anthropic,United States of America,Closed weights,2026


## 4. ECI — PCA Scatter

In [197]:
def run_eci_pca(dims=2):
    X = df_eci[ECI_BENCHMARKS].values.astype(float)
    X = SimpleImputer(strategy="mean").fit_transform(X)
    X = StandardScaler().fit_transform(X)

    n_comp = min(dims, X.shape[1], X.shape[0])
    pca = PCA(n_components=n_comp, random_state=42)
    coords = pca.fit_transform(X)

    df_out = df_eci.copy()
    df_out["PC1"] = coords[:, 0]
    df_out["PC2"] = coords[:, 1]
    if dims == 3 and n_comp >= 3:
        df_out["PC3"] = coords[:, 2]
    return df_out, pca.explained_variance_ratio_, pca

# ── 2D ────────────────────────────────────────────────────────────────────────
df_eci_pca, eci_var, _ = run_eci_pca(dims=2)

fig_eci_pca2d = px.scatter(
    df_eci_pca,
    x="PC1", y="PC2",
    color="Country",
    hover_name="Model",
    hover_data={"Organization": True, "Release Year": True, "eci": ":.1f", "PC1": ":.3f", "PC2": ":.3f"},
    title=f"ECI — PCA 2D<br><sup>PC1 {eci_var[0]:.1%} var | PC2 {eci_var[1]:.1%} var | {TOP_K_BENCHMARKS} benchmarks</sup>",
    labels={"PC1": f"PC1 ({eci_var[0]:.1%})", "PC2": f"PC2 ({eci_var[1]:.1%})"},
    template="plotly_white",
    width=1280, height=720,
)
fig_eci_pca2d.update_traces(marker=dict(size=8, opacity=0.85))
fig_eci_pca2d.show()

# ── 3D ────────────────────────────────────────────────────────────────────────
df_eci_pca3, eci_var3, _ = run_eci_pca(dims=3)

fig_eci_pca3d = px.scatter_3d(
    df_eci_pca3,
    x="PC1", y="PC2", z="PC3",
    color="Country",
    hover_name="Model",
    hover_data={"Organization": True, "Release Year": True, "eci": ":.1f"},
    title=f"ECI — PCA 3D<br><sup>PC1 {eci_var3[0]:.1%} | PC2 {eci_var3[1]:.1%} | PC3 {eci_var3[2]:.1%}</sup>",
    labels={
        "PC1": f"PC1 ({eci_var3[0]:.1%})",
        "PC2": f"PC2 ({eci_var3[1]:.1%})",
        "PC3": f"PC3 ({eci_var3[2]:.1%})",
    },
    template="plotly_white",
    width=1280, height=720,
)
fig_eci_pca3d.update_traces(marker=dict(size=5, opacity=0.85))
fig_eci_pca3d.show()

## 4b. ECI — PCA Loadings

In [198]:
# Refit PCA for loadings
X_eci_load = df_eci[ECI_BENCHMARKS].values.astype(float)
X_eci_load = SimpleImputer(strategy="mean").fit_transform(X_eci_load)
X_eci_load = StandardScaler().fit_transform(X_eci_load)

eci_pca_full = PCA(n_components=min(3, len(ECI_BENCHMARKS)), random_state=42)
eci_pca_full.fit(X_eci_load)
eci_var_full = eci_pca_full.explained_variance_ratio_

n_comp_eci = eci_pca_full.n_components_
df_eci_loadings = pd.DataFrame(
    eci_pca_full.components_[:n_comp_eci].T,
    index=ECI_BENCHMARKS,
    columns=[f"PC{i+1} ({eci_var_full[i]:.1%})" for i in range(n_comp_eci)],
).reset_index().rename(columns={"index": "Benchmark"})

df_eci_melt = df_eci_loadings.melt(id_vars="Benchmark", var_name="Component", value_name="Loading")

fig_eci_load = px.bar(
    df_eci_melt,
    x="Benchmark", y="Loading", color="Component",
    barmode="group",
    title="ECI — PCA Loadings per Benchmark",
    template="plotly_white",
    width=1280, height=720,
)
fig_eci_load.add_hline(y=0, line_color="black", line_width=0.5)
fig_eci_load.update_layout(xaxis_tickangle=-30)
fig_eci_load.show()

fig_eci_load_heat = go.Figure(go.Heatmap(
    z=eci_pca_full.components_[:n_comp_eci],
    x=ECI_BENCHMARKS,
    y=[f"PC{i+1} ({eci_var_full[i]:.1%})" for i in range(n_comp_eci)],
    colorscale="RdBu",
    zmid=0,
    text=np.round(eci_pca_full.components_[:n_comp_eci], 3),
    texttemplate="%{text}",
    hovertemplate="Benchmark: %{x}<br>Component: %{y}<br>Loading: %{z:.3f}<extra></extra>",
))
fig_eci_load_heat.update_layout(
    title="ECI — PCA Loadings Heatmap",
    xaxis_title="Benchmark",
    yaxis_title="Principal Component",
    template="plotly_white",
    xaxis_tickangle=-30,
    width=1280, height=720,
)
fig_eci_load_heat.show()

## 4c. ECI — UMAP

In [199]:
X_eci_umap = df_eci[ECI_BENCHMARKS].values.astype(float)
X_eci_umap = SimpleImputer(strategy="mean").fit_transform(X_eci_umap)
X_eci_umap = StandardScaler().fit_transform(X_eci_umap)

eci_reducer = umap.UMAP(n_components=2, n_neighbors=10, min_dist=0.1, random_state=42)
eci_embedding = eci_reducer.fit_transform(X_eci_umap)

df_eci["UMAP1"] = eci_embedding[:, 0]
df_eci["UMAP2"] = eci_embedding[:, 1]

fig_eci_umap = px.scatter(
    df_eci,
    x="UMAP1", y="UMAP2",
    color="Country",
    hover_name="Model",
    hover_data={
        "Organization": True,
        "Release Year": True,
        "eci": ":.1f",
        "UMAP1": False,
        "UMAP2": False,
        **{b: ":.3f" for b in ECI_BENCHMARKS},
    },
    title=f"ECI — UMAP 2D<br><sup>{TOP_K_BENCHMARKS} benchmarks, n_neighbors=10, min_dist=0.1</sup>",
    template="plotly_white",
    width=1280, height=720,
)
fig_eci_umap.update_traces(marker=dict(size=8, opacity=0.85))
fig_eci_umap.show()

C:\Users\aidan\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



## 4d. ECI — UMAP Dimension Correlation

In [200]:
eci_corr_rows = []
for bench in ECI_BENCHMARKS:
    vals = df_eci[bench].values.astype(float)
    mask = ~np.isnan(vals)
    r1, _ = spearmanr(vals[mask], df_eci["UMAP1"].values[mask])
    r2, _ = spearmanr(vals[mask], df_eci["UMAP2"].values[mask])
    eci_corr_rows.append({"Benchmark": bench, "UMAP1": round(r1, 3), "UMAP2": round(r2, 3)})

df_eci_corr = pd.DataFrame(eci_corr_rows)
df_eci_corr_melt = df_eci_corr.melt(id_vars="Benchmark", var_name="Dimension", value_name="Spearman r")

fig_eci_ucorr = px.bar(
    df_eci_corr_melt,
    x="Benchmark", y="Spearman r", color="Dimension",
    barmode="group",
    title="ECI — UMAP Dimension Correlation (Spearman r)",
    template="plotly_white",
    width=1280, height=720,
)
fig_eci_ucorr.add_hline(y=0, line_color="black", line_width=0.5)
fig_eci_ucorr.update_layout(xaxis_tickangle=-30, yaxis_range=[-1, 1])
fig_eci_ucorr.show()

fig_eci_ucorr_heat = go.Figure(go.Heatmap(
    z=df_eci_corr[["UMAP1", "UMAP2"]].values.T,
    x=df_eci_corr["Benchmark"].tolist(),
    y=["UMAP1", "UMAP2"],
    colorscale="RdBu",
    zmid=0, zmin=-1, zmax=1,
    text=df_eci_corr[["UMAP1", "UMAP2"]].values.T,
    texttemplate="%{text}",
    hovertemplate="Benchmark: %{x}<br>Dimension: %{y}<br>Spearman r: %{z:.3f}<extra></extra>",
))
fig_eci_ucorr_heat.update_layout(
    title="ECI — UMAP Dimension Correlation Heatmap",
    xaxis_title="Benchmark",
    yaxis_title="UMAP Dimension",
    template="plotly_white",
    xaxis_tickangle=-30,
    width=1280, height=720,
)
fig_eci_ucorr_heat.show()

## 5. ECI — Custom-Axis Scatter

In [201]:
# Pick any two benchmarks from ECI_BENCHMARKS (printed above)
ECI_X = ECI_BENCHMARKS[0]
ECI_Y = ECI_BENCHMARKS[1]
ECI_COLOR = "Country"  # "Country" | "Organization" | "Access"

df_eci_scatter = df_eci.dropna(subset=[ECI_X, ECI_Y])

fig_eci_scatter = px.scatter(
    df_eci_scatter,
    x=ECI_X, y=ECI_Y,
    color=ECI_COLOR,
    hover_name="Model",
    hover_data={"Organization": True, "Country": True, "Release Year": True, "eci": ":.1f",
                ECI_X: ":.3f", ECI_Y: ":.3f"},
    title=f"ECI — {ECI_Y} vs {ECI_X}",
    trendline="ols",
    template="plotly_white",
    width=1280, height=720,
)
fig_eci_scatter.update_traces(marker=dict(size=9, opacity=0.8))
fig_eci_scatter.show()

## 6. ECI — Classifier + Confusion Matrix + ROC

In [ ]:
ECI_CLASSIFIER  = "dt"      # "svm" | "dt" | "lr"
ECI_TARGET      = "country"  # "country" | "organization"
ECI_TOP_N       = 8

# ── Country filter ────────────────────────────────────────────────────────────
# None = all, or list specific values from the Country column
# Note: ECI uses "United States of America" (not "United States")
ECI_ORIGINS = ["United States of America", "China"]
# ─────────────────────────────────────────────────────────────────────────────

ECI_TARGET_MAP = {
    "country":      ("Country",      "Country"),
    "organization": ("Organization", "Organization"),
}
eci_col, eci_target_label = ECI_TARGET_MAP[ECI_TARGET]

df_eci_clf = df_eci.copy()
if ECI_ORIGINS is not None:
    df_eci_clf = df_eci_clf[df_eci_clf["Country"].isin(ECI_ORIGINS)]

eci_y_raw = df_eci_clf[eci_col].fillna("Unknown").astype(str)
eci_top = set(eci_y_raw.value_counts().head(ECI_TOP_N).index)
eci_mask = eci_y_raw.isin(eci_top)
eci_y_raw, df_eci_clf = eci_y_raw[eci_mask], df_eci_clf[eci_mask]
eci_cc = eci_y_raw.value_counts()
eci_valid = set(eci_cc[eci_cc >= 3].index)
eci_mask2 = eci_y_raw.isin(eci_valid)
eci_y_raw, df_eci_clf = eci_y_raw[eci_mask2], df_eci_clf[eci_mask2]

X_eci_clf = df_eci_clf[ECI_BENCHMARKS].values.astype(float)
X_eci_clf = SimpleImputer(strategy="mean").fit_transform(X_eci_clf)
X_eci_clf = StandardScaler().fit_transform(X_eci_clf)
y_eci = eci_y_raw.values
eci_labels = sorted(eci_y_raw.unique().tolist())

try:
    X_eci_tr, X_eci_te, y_eci_tr, y_eci_te = train_test_split(
        X_eci_clf, y_eci, test_size=0.25, random_state=42, stratify=y_eci)
except ValueError:
    X_eci_tr, X_eci_te, y_eci_tr, y_eci_te = train_test_split(
        X_eci_clf, y_eci, test_size=0.25, random_state=42)

eci_clf_map = {
    "svm": (SVC(kernel="rbf", C=1.0, random_state=42, probability=True), "SVM (RBF)"),
    "dt":  (DecisionTreeClassifier(max_depth=5, random_state=42),          "Decision Tree"),
    "lr":  (LogisticRegression(max_iter=2000, random_state=42),            "Logistic Regression"),
}
eci_clf, eci_clf_name = eci_clf_map[ECI_CLASSIFIER]
eci_clf.fit(X_eci_tr, y_eci_tr)
y_eci_pred = eci_clf.predict(X_eci_te)
y_eci_prob = eci_clf.predict_proba(X_eci_te)
eci_acc = accuracy_score(y_eci_te, y_eci_pred)
eci_cm = confusion_matrix(y_eci_te, y_eci_pred, labels=eci_labels)

eci_origin_label = f"{', '.join(ECI_ORIGINS)}" if ECI_ORIGINS else "All"
print(f"{eci_clf_name} | target: {eci_target_label} | origins: {eci_origin_label} | accuracy: {eci_acc:.1%} ({len(X_eci_te)} test samples)")

Logistic Regression | target: Country | origins: United States of America, China | accuracy: 64.3% (28 test samples)


In [206]:
# ── ECI Confusion Matrix ──────────────────────────────────────────────────────
eci_cm_norm = eci_cm.astype(float) / eci_cm.sum(axis=1, keepdims=True).clip(min=1)

fig_eci_cm = go.Figure(go.Heatmap(
    z=eci_cm_norm,
    x=eci_labels,
    y=eci_labels,
    colorscale="Blues",
    zmin=0, zmax=1,
    text=eci_cm,
    texttemplate="%{text}",
    hovertemplate="True: %{y}<br>Pred: %{x}<br>Count: %{text}<br>Rate: %{z:.1%}<extra></extra>",
))
fig_eci_cm.update_layout(
    title=f"ECI — Confusion Matrix — {eci_clf_name} predicting {eci_target_label}<br><sup>Accuracy: {eci_acc:.1%}</sup>",
    xaxis_title="Predicted",
    yaxis_title="True",
    template="plotly_white",
    yaxis_autorange="reversed",
    width=900, height=720,
)
fig_eci_cm.show()

# ── ECI ROC AUC ──────────────────────────────────────────────────────────────
y_eci_te_bin = label_binarize(y_eci_te, classes=eci_labels)

fig_eci_roc = go.Figure()
eci_aucs = {}

if len(eci_labels) == 2:
    fpr, tpr, _ = roc_curve(y_eci_te_bin[:, 0],
                            y_eci_prob[:, 0] if y_eci_prob.shape[1] == 1 else y_eci_prob[:, 1])
    roc_auc = auc(fpr, tpr)
    eci_aucs[eci_labels[1]] = roc_auc
    fig_eci_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines",
        name=f"{eci_labels[1]} vs {eci_labels[0]} (AUC={roc_auc:.3f})",
    ))
else:
    for i, lbl in enumerate(eci_labels):
        fpr, tpr, _ = roc_curve(y_eci_te_bin[:, i], y_eci_prob[:, i])
        roc_auc = auc(fpr, tpr)
        eci_aucs[lbl] = roc_auc
        fig_eci_roc.add_trace(go.Scatter(
            x=fpr, y=tpr, mode="lines",
            name=f"{lbl} (AUC={roc_auc:.3f})",
        ))

fig_eci_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines",
    line=dict(dash="dash", color="gray"),
    name="Random (AUC=0.500)", showlegend=True,
))

eci_macro_auc = np.mean(list(eci_aucs.values()))
fig_eci_roc.update_layout(
    title=f"ECI — ROC Curve — {eci_clf_name} predicting {eci_target_label}<br><sup>Macro AUC: {eci_macro_auc:.3f}</sup>",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
    width=800, height=800,
    xaxis=dict(constrain="domain", scaleanchor="y"),
    yaxis=dict(constrain="domain"),
    legend=dict(x=0.6, y=0.05),
)
fig_eci_roc.show()

In [204]:
# ── ECI Feature importances (DT / LR only) ───────────────────────────────────
if ECI_CLASSIFIER == "dt":
    eci_importances = dict(zip(ECI_BENCHMARKS, eci_clf.feature_importances_))
elif ECI_CLASSIFIER == "lr" and hasattr(eci_clf, "coef_"):
    eci_importances = dict(zip(ECI_BENCHMARKS, np.abs(eci_clf.coef_).mean(axis=0)))
else:
    eci_importances = None

if eci_importances:
    df_eci_imp = pd.DataFrame(list(eci_importances.items()), columns=["Benchmark", "Importance"])
    df_eci_imp = df_eci_imp.sort_values("Importance", ascending=True)
    fig_eci_imp = px.bar(
        df_eci_imp, x="Importance", y="Benchmark", orientation="h",
        title=f"ECI — Feature Importances — {eci_clf_name}",
        template="plotly_white",
        width=1280, height=720,
    )
    fig_eci_imp.show()
else:
    print("SVM has no native feature importances — skipping.")

SVM has no native feature importances — skipping.
